# A-Share Risk-Return Dashboard — Analytical Workflow

**Module**: ACC102 Mini Assignment | Track 4 — Interactive Data Analysis Tool  
**Product**: A-Share Risk-Return Dashboard  
**Data Source**: Live OHLCV data from Tencent Finance (primary) / EastMoney Finance (fallback)  
**Date Accessed**: April 2026  

---

## Analytical Problem

How can a beginner investor or finance student quickly compare the **risk and return** of multiple A-share stocks without needing advanced financial knowledge?

This notebook demonstrates the complete analytical workflow behind the Streamlit dashboard:  
**data acquisition → data cleaning → feature engineering → metric calculation → visualisation → interpretation**

**Target User**: Non-technical investors and finance students who want a quick, clear view of A-share stock performance.

## 1. Setup & Imports

In [16]:
import os
import time
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

All libraries loaded successfully.


## 2. Data Acquisition

This project fetches **live OHLCV data** from Tencent Finance (primary source) with automatic fallback to EastMoney Finance — no local CSV files required for price data.

Each stock returns 6 fields per trading day:  
`date`, `open`, `high`, `low`, `close`, `volume`

We select 5 representative A-share stocks as our analysis sample.

In [17]:
# --- Configuration ---
TICKERS    = ['000001', '000002', '600519', '300750', '601318']
EXCHANGES  = {'000001':'SZ','000002':'SZ','600519':'SH','300750':'SZ','601318':'SH'}
START_DATE = '2022-01-01'
END_DATE   = '2026-04-14'

TENCENT_URL  = 'https://web.ifzq.gtimg.cn/appstock/app/fqkline/get'
EASTMONEY_URL = 'https://push2his.eastmoney.com/api/qt/stock/kline/get'
EM_UT = 'fa5fd1943c7b386f172d6893dbfba10b'

session = requests.Session()
session.trust_env = False   # bypass system proxy
session.headers.update({
    'User-Agent': 'Mozilla/5.0',
    'Referer': 'https://quote.eastmoney.com/',
})

def fetch_tencent(symbol, exchange, start, end):
    tsym  = f"sz{symbol}" if exchange == 'SZ' else f"sh{symbol}"
    param = f"{tsym},day,{start},{end},640,qfq"
    resp  = session.get(TENCENT_URL, params={'param': param}, timeout=15)
    resp.raise_for_status()
    data   = resp.json().get('data') or {}
    klines = (data.get(tsym) or {})
    klines = klines.get('qfqday') or klines.get('day') or []
    return [{'date':str(k[0]),'open':str(k[1]),'high':str(k[3]),
             'low':str(k[4]),'close':str(k[2]),'volume':str(k[5])} for k in klines if len(k)>=6]

def fetch_eastmoney(symbol, exchange, start, end):
    secid  = f"0.{symbol}" if exchange == 'SZ' else f"1.{symbol}"
    beg, e = start.replace('-',''), end.replace('-','')
    params = {'secid':secid,'ut':EM_UT,'fields1':'f1,f2,f3,f4,f5,f6',
              'fields2':'f51,f52,f53,f54,f55,f56,f57,f58','klt':'101','fqt':'0','beg':beg,'end':e}
    resp   = session.get(EASTMONEY_URL, params=params, timeout=15)
    resp.raise_for_status()
    klines = (resp.json().get('data') or {}).get('klines') or []
    rows = []
    for k in klines:
        p = k.split(',')
        if len(p) >= 6:
            rows.append({'date':p[0],'open':p[1],'high':p[3],'low':p[4],'close':p[2],'volume':p[5]})
    return rows

def fetch_ohlcv(symbol, exchange, start, end):
    rows = []
    try:
        rows = fetch_tencent(symbol, exchange, start, end)
    except Exception:
        pass
    if not rows:
        rows = fetch_eastmoney(symbol, exchange, start, end)
    if not rows:
        raise ValueError(f"No data for {symbol}")
    dedup = {r['date']: r for r in rows}
    rows  = [dedup[d] for d in sorted(dedup)]
    df = pd.DataFrame(rows)
    df['date']   = pd.to_datetime(df['date'], errors='coerce')
    for col in ['open','high','low','close','volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['date','close']).drop_duplicates('date').sort_values('date')
    return df.set_index('date')[['open','high','low','close','volume']]

raw_frames = {}
for t in TICKERS:
    try:
        raw_frames[t] = fetch_ohlcv(t, EXCHANGES[t], START_DATE, END_DATE)
        df = raw_frames[t]
        print(f"  {t}: {len(df)} rows, {df.index.min().date()} → {df.index.max().date()}")
        time.sleep(0.1)
    except Exception as e:
        print(f"  ⚠ {t}: {e}")

print(f'\nLoaded {len(raw_frames)} stocks')

  000001: 640 rows, 2023-08-18 → 2026-04-14
  000002: 640 rows, 2023-08-18 → 2026-04-14
  600519: 640 rows, 2023-08-18 → 2026-04-14
  300750: 640 rows, 2023-08-18 → 2026-04-14
  601318: 640 rows, 2023-08-18 → 2026-04-14

Loaded 5 stocks


## 3. Data Cleaning & Quality Report

The following cleaning steps are applied to the raw API response before analysis:

1. **Unified column names** — all lowercase
2. **Date parsing** — convert to `datetime`, drop rows with invalid dates (`NaT`)
3. **Numeric conversion** — coerce non-numeric price/volume values to `NaN`
4. **Remove invalid prices** — drop rows where `close ≤ 0`
5. **Structural sanity check** — drop rows where `high < low`
6. **Deduplication** — keep one record per date
7. **Forward-fill** — fill gaps from non-trading days when building the price matrix

In [18]:
# Build close-price matrix
prices = pd.concat(
    [df['close'].rename(code) for code, df in raw_frames.items()], axis=1
).sort_index()

print('=== Missing Values — Before Cleaning ===')
print(prices.isna().sum())
print(f'Rows before cleaning: {len(prices)}')

# Forward-fill + drop all-NaN rows
prices = prices.ffill().dropna(how='all')

print('\n=== Missing Values — After Cleaning ===')
print(prices.isna().sum())
print(f'Rows after cleaning: {len(prices)}')
print(f'\nDate range: {prices.index.min().date()} → {prices.index.max().date()}')
print(f'Total trading days: {len(prices)}')
prices.tail()

=== Missing Values — Before Cleaning ===
000001    0
000002    0
600519    0
300750    0
601318    0
dtype: int64
Rows before cleaning: 640

=== Missing Values — After Cleaning ===
000001    0
000002    0
600519    0
300750    0
601318    0
dtype: int64
Rows after cleaning: 640

Date range: 2023-08-18 → 2026-04-14
Total trading days: 640


,000001,000002,600519,300750,601318
date,,,,,
2026-04-08,11.22,3.95,1465.02,391.30,59.45
2026-04-09,11.10,3.87,1460.49,389.99,58.74
2026-04-10,11.09,3.89,1453.96,416.00,58.82
2026-04-13,11.07,3.91,1443.31,427.13,57.75
2026-04-14,11.17,4.02,1446.90,430.79,58.97


In [19]:
print('=== Descriptive Statistics (Closing Prices) ===')
prices.describe().round(2)

=== Descriptive Statistics (Closing Prices) ===


,000001,000002,600519,300750,601318
count,640.00,640.00,640.00,640.00,640.00
mean,10.27,7.79,1482.14,247.82,48.22
std,1.26,2.29,100.72,79.55,10.11
min,7.47,3.81,1185.56,128.97,32.82
25%,9.24,6.50,1408.97,178.84,38.10
50%,10.78,7.08,1460.72,244.66,48.64
75%,11.22,9.21,1554.07,279.32,56.58
max,12.94,14.20,1740.58,430.79,74.32


## 4. Feature Engineering — Daily Returns

Daily return formula:

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

Daily returns are the foundation for all risk-return metrics.

In [20]:
returns = prices.pct_change().dropna()

print('=== Daily Returns — Descriptive Statistics (%) ===')
(returns * 100).describe().round(4)

=== Daily Returns — Descriptive Statistics (%) ===


,000001,000002,600519,300750,601318
count,639.0000,639.0000,639.0000,639.0000,639.0000
mean,0.0275,-0.1625,-0.0116,0.1349,0.0693
std,1.4283,2.4016,1.5280,2.5766,1.7267
min,-9.9701,-10.0193,-7.7600,-15.0563,-10.3512
25%,-0.6948,-1.4862,-0.7079,-1.1503,-0.8060
50%,-0.0876,-0.3021,-0.1183,-0.2334,-0.0182
75%,0.6459,0.6501,0.5052,1.1976,0.8191
max,11.8687,10.0629,9.8220,19.2187,10.7178


## 5. Metric Calculation

We compute 8 risk-return metrics:

| Metric | Formula |
|---|---|
| Total Return | P_end / P_start − 1 |
| Annualised Return | (1 + Total Return)^(252/N) − 1 |
| Annualised Volatility | σ_daily × √252 |
| Max Drawdown | min(P_t / cummax(P) − 1) |
| Sharpe Ratio | Ann. Return / Ann. Volatility |
| Calmar Ratio | Ann. Return / |Max Drawdown| |
| Win Rate | % days with positive return |
| Avg Up/Down Day | mean return on positive/negative days |

In [21]:
def calculate_metrics(prices_df):
    rets = prices_df.pct_change().dropna()
    rows = []
    for col in prices_df.columns:
        p = prices_df[col].dropna()
        r = rets[col].dropna()
        N = len(p)

        total_ret  = p.iloc[-1] / p.iloc[0] - 1
        ann_ret    = (1 + total_ret) ** (252 / N) - 1
        ann_vol    = r.std() * np.sqrt(252)
        drawdown   = p / p.cummax() - 1
        max_dd     = drawdown.min()
        sharpe     = ann_ret / ann_vol if ann_vol != 0 else np.nan
        calmar     = ann_ret / abs(max_dd) if max_dd != 0 else np.nan
        win_rate   = (r > 0).mean()
        avg_up     = r[r > 0].mean() if (r > 0).any() else 0
        avg_dn     = r[r < 0].mean() if (r < 0).any() else 0

        rows.append({
            'Ticker'           : col,
            'Total Return'     : total_ret,
            'Ann. Return'      : ann_ret,
            'Ann. Volatility'  : ann_vol,
            'Max Drawdown'     : max_dd,
            'Sharpe Ratio'     : sharpe,
            'Calmar Ratio'     : calmar,
            'Win Rate'         : win_rate,
            'Avg Up Day'       : avg_up,
            'Avg Down Day'     : avg_dn,
        })
    return pd.DataFrame(rows).set_index('Ticker')


metrics = calculate_metrics(prices)

pct_cols = ['Total Return','Ann. Return','Ann. Volatility',
            'Max Drawdown','Win Rate','Avg Up Day','Avg Down Day']
metrics.style.format({c: '{:.2%}' for c in pct_cols} |
                     {'Sharpe Ratio': '{:.3f}', 'Calmar Ratio': '{:.3f}'})\
             .background_gradient(subset=['Total Return'], cmap='RdYlGn')\
             .background_gradient(subset=['Sharpe Ratio'], cmap='RdYlGn')\
             .background_gradient(subset=['Max Drawdown'], cmap='RdYlGn_r')

,Total Return,Ann. Return,Ann. Volatility,Max Drawdown,Sharpe Ratio,Calmar Ratio,Win Rate,Avg Up Day,Avg Down Day
Ticker,,,,,,,,,
000001,11.73%,4.47%,22.67%,-25.31%,0.197,0.176,46.17%,1.05%,-0.90%
000002,-70.52%,-38.18%,38.13%,-73.17%,-1.001,-0.522,41.16%,1.83%,-1.67%
600519,-13.73%,-5.65%,24.26%,-31.89%,-0.233,-0.177,44.60%,1.09%,-0.90%
300750,92.24%,29.35%,40.90%,-43.11%,0.718,0.681,44.91%,2.07%,-1.44%
601318,41.63%,14.69%,27.41%,-27.98%,0.536,0.525,48.83%,1.29%,-1.12%


## 6. Visualisation

### 6.1 Normalised Price Chart (Base = 100)

In [22]:
norm = prices / prices.iloc[0] * 100
fig = px.line(norm, title='Normalised Price (Start = 100)',
              labels={'value': 'Indexed Price', 'variable': 'Ticker'})
fig.add_hline(y=100, line_dash='dot', line_color='grey')
fig.show()

### 6.2 Drawdown Curves

In [23]:
dd_df = pd.DataFrame()
for col in prices.columns:
    dd_df[col] = (prices[col] / prices[col].cummax() - 1) * 100

fig = px.area(dd_df, title='Drawdown Curves (%)',
              labels={'value': 'Drawdown (%)', 'variable': 'Ticker'})
fig.show()

### 6.3 Risk-Return Scatter Plot

In [24]:
scatter_df = metrics[['Ann. Return', 'Ann. Volatility', 'Sharpe Ratio']].copy()
scatter_df['Ann. Return (%)']    = scatter_df['Ann. Return'] * 100
scatter_df['Ann. Volatility (%)'] = scatter_df['Ann. Volatility'] * 100
scatter_df = scatter_df.reset_index()

fig = px.scatter(
    scatter_df,
    x='Ann. Volatility (%)', y='Ann. Return (%)',
    text='Ticker', color='Sharpe Ratio',
    size=[20]*len(scatter_df),
    color_continuous_scale='RdYlGn',
    title='Risk-Return Scatter (colour = Sharpe Ratio)',
)
fig.update_traces(textposition='top center')
fig.show()

### 6.4 Correlation Heatmap

In [25]:
corr = returns.corr()
fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu',
                zmin=-1, zmax=1, title='Return Correlation Heatmap')
fig.show()

### 6.5 Rolling 30-Day Annualised Volatility

In [26]:
roll_vol = returns.rolling(30).std() * np.sqrt(252) * 100
fig = px.line(roll_vol, title='Rolling 30-Day Annualised Volatility (%)',
              labels={'value': 'Volatility (%)', 'variable': 'Ticker'})
fig.show()

### 6.6 Daily Returns Distribution (Violin Plot)

In [27]:
fig = px.violin(
    returns * 100,
    title='Daily Returns Distribution (%)',
    box=True, points=False,
    labels={'variable': 'Ticker', 'value': 'Daily Return (%)'},
)
fig.show()

### 6.7 Monthly Returns Heatmap

In [28]:
def monthly_heatmap(prices_series, ticker_name):
    r = prices_series.resample('ME').last().pct_change().dropna() * 100
    df = pd.DataFrame({'year': r.index.year, 'month': r.index.month, 'ret': r.values})
    pivot = df.pivot_table(index='year', columns='month', values='ret')
    month_names = ['Jan','Feb','Mar','Apr','May','Jun',
                   'Jul','Aug','Sep','Oct','Nov','Dec']
    pivot.columns = [month_names[m-1] for m in pivot.columns]
    fig = px.imshow(pivot, text_auto='.1f', color_continuous_scale='RdYlGn',
                    zmin=-15, zmax=15,
                    title=f'Monthly Returns Heatmap — {ticker_name} (%)')
    fig.show()

monthly_heatmap(prices['000001'], '000001')

## 7. Key Insights & Interpretation

In [29]:
best_ret    = metrics['Total Return'].idxmax()
lowest_vol  = metrics['Ann. Volatility'].idxmin()
worst_dd    = metrics['Max Drawdown'].idxmin()
best_sharpe = metrics['Sharpe Ratio'].idxmax()

print('=== Key Insights ===')
print(f"Top performer   : {best_ret} ({metrics.loc[best_ret,'Total Return']*100:.1f}% total return)")
print(f"Lowest risk     : {lowest_vol} ({metrics.loc[lowest_vol,'Ann. Volatility']*100:.1f}% volatility)")
print(f"Largest drawdown: {worst_dd} ({metrics.loc[worst_dd,'Max Drawdown']*100:.1f}%)")
print(f"Best risk-adjusted: {best_sharpe} (Sharpe {metrics.loc[best_sharpe,'Sharpe Ratio']:.2f})")

print('\n=== Interpretation ===')
print(f"""
Over the analysis period, {best_ret} delivered the highest total return,
but with higher volatility and deeper drawdown.
{lowest_vol} showed the most stable profile with the lowest annualised volatility,
which may be more suitable for risk-averse investors.
On a risk-adjusted basis, {best_sharpe} achieved the highest Sharpe ratio
({metrics.loc[best_sharpe,'Sharpe Ratio']:.2f}), indicating the strongest return per unit of risk.
Therefore, investment decisions should combine return, volatility, drawdown, and Sharpe,
rather than relying on absolute return alone.
""")

=== Key Insights ===
Top performer   : 300750 (92.2% total return)
Lowest risk     : 000001 (22.7% volatility)
Largest drawdown: 000002 (-73.2%)
Best risk-adjusted: 300750 (Sharpe 0.72)

=== Interpretation ===

Over the analysis period, 300750 delivered the highest total return,
but with higher volatility and deeper drawdown.
000001 showed the most stable profile with the lowest annualised volatility,
which may be more suitable for risk-averse investors.
On a risk-adjusted basis, 300750 achieved the highest Sharpe ratio
(0.72), indicating the strongest return per unit of risk.
Therefore, investment decisions should combine return, volatility, drawdown, and Sharpe,
rather than relying on absolute return alone.



## 8. Save Analysis Results (Optional)

In [30]:
os.makedirs('../data', exist_ok=True)

prices.to_csv('../data/sample_prices.csv')
metrics.to_csv('../data/sample_metrics.csv')
print('Analysis results saved to ../data/')

Analysis results saved to ../data/


## 9. Limitations & Reliability Notes

- **Risk-free rate is assumed as 0**. In practice, Sharpe should use a proper benchmark risk-free rate.
- **No transaction costs** are modelled. Real-world returns would be lower after fees, taxes, and slippage.
- **Daily frequency only** — intraday risk is not captured.
- **Survivorship bias** may exist because delisted stocks are not included in the selected sample.
- **Data quality** issues may still exist around corporate actions or suspended trading periods.
- **Past performance does not guarantee future returns**. Results are for educational use only.

---

**Data Source**: Tencent Finance / EastMoney Finance live API (with fallback)  
**Date Accessed**: April 2026  
**This notebook is for ACC102 coursework only and does not constitute investment advice.**